In [1]:
# ==============================================================================
# BLOCK 1 — ENVIRONMENT SETUP AND INPUT FILE INSPECTION
# ==============================================================================

import os

print("🔍 INSPECTING INPUT FILES IN /kaggle/input/ ...\n")

input_dir = "/kaggle/input/"
found_files = []

for root, dirs, files in os.walk(input_dir):
    for file in files:
        full_path = os.path.join(root, file)
        found_files.append(os.path.join(root, file))
        print(f"input_file_path = '{os.path.join(root, file)}'")

if not found_files:
    raise FileNotFoundError(
        "No files found in /kaggle/input/. "
        "Make sure the Kaggle dataset is attached to the notebook."
    )

print("\n" + "=" * 60)
print("📁 WORKING DIRECTORY FOR OUTPUTS:")
working_dir = "/kaggle/working/"
print(f"output_file_path = "
      f"'{os.path.join(working_dir, 'SMILES_CHIEFPTML_with_fchiral.xlsx')}'")

# Install RDKit
!pip install -q rdkit

from rdkit import Chem
from rdkit.Chem import inchi

print("\n✅ RDKit successfully imported.")

🔍 INSPECTING INPUT FILES IN /kaggle/input/ ...

input_file_path = '/kaggle/input/datasets/maiderb/smiles-chiefptml2/SMILES.xlsx'

📁 WORKING DIRECTORY FOR OUTPUTS:
output_file_path = '/kaggle/working/SMILES_CHIEFPTML_with_fchiral.xlsx'

✅ RDKit successfully imported.


In [2]:
# ==============================================================================
# BLOCK 2 — CHIRALITY SCORE CORE FUNCTIONS
# ==============================================================================

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import math
import os
from typing import List, Dict, Tuple, Optional

import pandas as pd
import numpy as np

try:
    from rdkit import Chem
    from rdkit.Chem import inchi as rd_inchi
except ImportError:
    raise SystemExit("RDKit is required in your environment.")

SCORE_MAP = {"R": 1.0, "S": 0.5, "U": 0.75, "N": 0.0}


def smiles_to_mol(smiles: str) -> Optional[Chem.Mol]:
    """Convert SMILES to RDKit Mol object safely. Return None if invalid."""
    try:
        mol = Chem.MolFromSmiles(smiles, sanitize=True)
        if mol is None:
            return None

        Chem.AssignStereochemistry(
            mol,
            cleanIt=True,
            force=True,
            flagPossibleStereoCenters=True
        )

        return mol

    except Exception:
        return None


def mol_to_inchi(mol: Chem.Mol) -> str:
    """Convert RDKit Mol object to IUPAC InChI string."""
    try:
        return rd_inchi.MolToInchi(mol)
    except Exception:
        return ""


def label_atoms(mol: Chem.Mol) -> Tuple[List[str], List[int], List[int]]:
    """
    Assign CIP stereochemical labels and 1-based canonical atomic positions.

    Returns:
        labels: List of atom labels (R, S, U, or N)
        center_indices: 0-based atom indices for stereocenters
        positions_1based: 1-based positions of stereocenters
    """

    n = mol.GetNumAtoms()
    labels = ["N"] * n

    chiral_list = Chem.FindMolChiralCenters(
        mol,
        includeUnassigned=True,
        useLegacyImplementation=False
    )

    for idx, _ in chiral_list:

        atom = mol.GetAtomWithIdx(idx)

        cip = (
            atom.GetProp("_CIPCode")
            if atom.HasProp("_CIPCode")
            else None
        )

        if cip == "R":
            labels[idx] = "R"

        elif cip == "S":
            labels[idx] = "S"

        elif cip == "?":
            labels[idx] = "U"

        else:
            labels[idx] = "N"

    center_indices = sorted(
        idx for idx in range(n)
        if labels[idx] in ("R", "S", "U")
    )

    positions_1based = [i + 1 for i in center_indices]

    return labels, center_indices, positions_1based


def compute_q_values(labels: List[str]) -> Tuple[float, float, float]:

    q0 = q1 = q2 = 0.0

    for i, lab in enumerate(labels):

        x = float(i + 1)
        s = SCORE_MAP.get(lab, 0.0)

        q0 += s

        if lab in ("R", "S", "U"):
            q1 += x

        q2 += x * s

    return q0, q1, q2


def f_chiral_from_q(q0: float, q1: float, q2: float) -> float:

    if q0 == 0.0 or q1 == 0.0:
        return 0.0

    return 1.0 / (1.0 + math.exp(-(q2 / q1)))


def process_record(chembl_id: str, smiles: str) -> Dict[str, object]:

    if (
        (str(chembl_id) or "").strip().lower() == "m.d."
        or
        (str(smiles) or "").strip().lower() == "m.d."
    ):

        return {
            "inchi": "",
            "absolute_configuration": "",
            "stereo_positions": "",
            "q0": 0.0,
            "q1": 0.0,
            "q2": 0.0,
            "f_chiral": 0.25,
            "observations": "m.d."
        }

    mol = smiles_to_mol(smiles)

    if mol is None:

        return {
            "inchi": "",
            "absolute_configuration": "",
            "stereo_positions": "",
            "q0": 0.0,
            "q1": 0.0,
            "q2": 0.0,
            "f_chiral": 0.25,
            "observations": "Invalid SMILES"
        }

    inchi = mol_to_inchi(mol)

    labels, center_indices, pos_1based = label_atoms(mol)

    config_centers = [
        labels[i]
        for i in center_indices
    ]

    q0, q1, q2 = compute_q_values(labels)

    f_chiral = (
        0.0
        if q0 == 0.0
        else f_chiral_from_q(q0, q1, q2)
    )

    return {
        "inchi": inchi,
        "absolute_configuration": ",".join(config_centers)
        if config_centers else "",

        "stereo_positions": ",".join(
            str(x) for x in pos_1based
        ) if pos_1based else "",

        "q0": q0,
        "q1": q1,
        "q2": q2,
        "f_chiral": f_chiral,
        "observations": "OK"
    }


print("✅ Chirality calculation core functions successfully loaded.")

✅ Chirality calculation core functions successfully loaded.


In [3]:
# ==============================================================================
# BLOCK 3 — LOAD DATASET AND CALCULATE f_chiral
# ==============================================================================

input_file_path = (
    "/kaggle/input/datasets/maiderb/"
    "smiles-chiefptml2/SMILES.xlsx"
)

output_file_path = (
    "/kaggle/working/"
    "SMILES_CHIEFPTML_with_fchiral.xlsx"
)

print("Input file:")
print(input_file_path)

# Read dataset
df = pd.read_excel(input_file_path)

print("\nDataset shape:", df.shape)

print("\nAvailable columns:")
for col in df.columns:
    print(" -", col)

# Identify compound ID column
if "CMPD_CHEMBLID" in df.columns:
    id_col = "CMPD_CHEMBLID"
elif "Fila ID" in df.columns:
    id_col = "Fila ID"
else:
    raise KeyError(
        "The dataset must contain either "
        "'CMPD_CHEMBLID' or 'Fila ID'."
    )

# Identify SMILES column
smiles_col = "CANONICAL_SMILES"

if smiles_col not in df.columns:
    raise KeyError(
        f"Required SMILES column '{smiles_col}' was not found."
    )

print("\nUsing:")
print("ID column:", id_col)
print("SMILES column:", smiles_col)

# Calculate chirality descriptors
results = []

for _, row in df.iterrows():

    result = process_record(
        row[id_col],
        row[smiles_col]
    )

    results.append(result)

df_results = pd.DataFrame(results)

# Append results to original dataframe
df_out = pd.concat(
    [df.reset_index(drop=True),
     df_results.reset_index(drop=True)],
    axis=1
)

# Save final dataset
df_out.to_excel(
    output_file_path,
    index=False
)

print("\n" + "=" * 70)
print("CALCULATION COMPLETED")
print("=" * 70)

print("Rows processed:", len(df_out))
print("Output file:")
print(output_file_path)

print("\nObservations:")
print(df_out["observations"].value_counts(dropna=False))

Input file:
/kaggle/input/datasets/maiderb/smiles-chiefptml2/SMILES.xlsx

Dataset shape: (65826, 2)

Available columns:
 - CMPD_CHEMBLID
 - CANONICAL_SMILES

Using:
ID column: CMPD_CHEMBLID
SMILES column: CANONICAL_SMILES


[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Charges were rearranged

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Charges were rearranged

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefined stereo

[17:42:16] WARNING: Omitted undefine


CALCULATION COMPLETED
Rows processed: 65826
Output file:
/kaggle/working/SMILES_CHIEFPTML_with_fchiral.xlsx

Observations:
observations
OK                65534
m.d.                291
Invalid SMILES        1
Name: count, dtype: int64


In [4]:
# ==============================================================================
# BLOCK 4 — PREPARE ROBUSTNESS BENCHMARK SAMPLE
# ==============================================================================

# Select valid chiral compounds
df_chiral = df_out[
    (df_out["f_chiral"] > 0) &
    (df_out["f_chiral"] != 0.25)
].drop_duplicates(
    subset=[smiles_col]
)

# Reproducible sample of up to 500 molecules
chiral_sample = df_chiral.sample(
    n=min(500, len(df_chiral)),
    random_state=42
).copy()

print("Total valid chiral compounds available:", len(df_chiral))
print("Compounds selected for robustness benchmark:", len(chiral_sample))

Total valid chiral compounds available: 19579
Compounds selected for robustness benchmark: 500


In [5]:
# ==============================================================================
# BLOCK 5 — ROBUSTNESS TO SMILES RANDOMIZATION
# ==============================================================================

import scipy.stats as stats

N_RANDOM_SMILES = 10

randomization_results = []

for _, row in chiral_sample.iterrows():

    smiles = row[smiles_col]

    mol = smiles_to_mol(smiles)

    if mol is None:
        continue

    original_f = row["f_chiral"]

    randomized_values = []

    for _ in range(N_RANDOM_SMILES):

        random_smiles = Chem.MolToSmiles(
            mol,
            doRandom=True
        )

        random_mol = smiles_to_mol(random_smiles)

        if random_mol is None:
            continue

        labels, _, _ = label_atoms(random_mol)

        q0, q1, q2 = compute_q_values(labels)

        f_random = (
            0.0
            if q0 == 0.0
            else f_chiral_from_q(q0, q1, q2)
        )

        randomized_values.append(f_random)

    if randomized_values:

        randomization_results.append({
            "original_f_chiral": original_f,
            "mean_randomized_f_chiral":
                np.mean(randomized_values),
            "std_randomized_f_chiral":
                np.std(randomized_values),
            "min_randomized_f_chiral":
                np.min(randomized_values),
            "max_randomized_f_chiral":
                np.max(randomized_values),
            "max_absolute_variation":
                np.max(
                    np.abs(
                        np.array(randomized_values)
                        - original_f
                    )
                )
        })

df_random = pd.DataFrame(randomization_results)

pearson_r, pearson_p = stats.pearsonr(
    df_random["original_f_chiral"],
    df_random["mean_randomized_f_chiral"]
)

print("\n" + "=" * 70)
print("SMILES RANDOMIZATION ROBUSTNESS")
print("=" * 70)

print("Molecules evaluated:", len(df_random))
print("Randomized SMILES per molecule:", N_RANDOM_SMILES)

print(f"\nPearson correlation: {pearson_r:.6f}")

print(
    "Mean absolute variation:",
    f"{df_random['max_absolute_variation'].mean():.6e}"
)

print(
    "Median absolute variation:",
    f"{df_random['max_absolute_variation'].median():.6e}"
)

print(
    "Maximum absolute variation:",
    f"{df_random['max_absolute_variation'].max():.6e}"
)

randomization_file = (
    "/kaggle/working/"
    "smiles_randomization_robustness.csv"
)

df_random.to_csv(
    randomization_file,
    index=False
)

print("\nResults saved to:")
print(randomization_file)


SMILES RANDOMIZATION ROBUSTNESS
Molecules evaluated: 500
Randomized SMILES per molecule: 10

Pearson correlation: 0.966982
Mean absolute variation: 1.548307e-02
Median absolute variation: 0.000000e+00
Maximum absolute variation: 9.500271e-02

Results saved to:
/kaggle/working/smiles_randomization_robustness.csv


In [7]:
# ==============================================================================
# BLOCK 6 — RDKit MULTI-VERSION ROBUSTNESS BENCHMARK
# ==============================================================================
# Each RDKit version is executed in an independent Python subprocess.
# The scientific definition of f_chiral is kept unchanged.
# ==============================================================================

import os
import sys
import subprocess
import pandas as pd
import numpy as np


# ------------------------------------------------------------------------------
# 1. Reference/current RDKit
# ------------------------------------------------------------------------------

from rdkit import rdBase

current_rdkit_version = rdBase.rdkitVersion

print("Current Python version:")
print(sys.version)

print("\nCurrent RDKit version:")
print(current_rdkit_version)


# ------------------------------------------------------------------------------
# 2. Versions to compare
# ------------------------------------------------------------------------------

alternative_versions = [
    "2024.3.1",
    "2024.9.1",
    "2025.3.1"
]

print("\nRDKit versions selected for comparison:")

for version in alternative_versions:
    print(" -", version)


# ------------------------------------------------------------------------------
# 3. Prepare exactly the same 500 molecules
# ------------------------------------------------------------------------------

benchmark_smiles = (
    chiral_sample[smiles_col]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

benchmark_smiles = benchmark_smiles[:500]

print("\nBenchmark molecules exported:", len(benchmark_smiles))


# ------------------------------------------------------------------------------
# 4. Save benchmark input
# ------------------------------------------------------------------------------

benchmark_file = (
    "/kaggle/working/"
    "f_chiral_500_benchmark_smiles.csv"
)

pd.DataFrame({
    "SMILES": benchmark_smiles
}).to_csv(
    benchmark_file,
    index=False
)


# ------------------------------------------------------------------------------
# 5. Create independent worker script
# ------------------------------------------------------------------------------

script_file = (
    "/kaggle/working/"
    "rdkit_version_worker.py"
)

worker_script = r'''
import sys
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit import rdBase


input_file = sys.argv[1]
output_file = sys.argv[2]


SCORE_MAP = {
    "R": 1.0,
    "S": 0.5,
    "U": 0.75,
    "N": 0.0
}


def calculate_f_chiral(smiles):

    try:

        mol = Chem.MolFromSmiles(
            smiles,
            sanitize=True
        )

        if mol is None:
            return 0.25

        Chem.AssignStereochemistry(
            mol,
            cleanIt=True,
            force=True,
            flagPossibleStereoCenters=True
        )

        labels = ["N"] * mol.GetNumAtoms()

        chiral_list = Chem.FindMolChiralCenters(
            mol,
            includeUnassigned=True,
            useLegacyImplementation=False
        )

        for idx, _ in chiral_list:

            atom = mol.GetAtomWithIdx(idx)

            cip = (
                atom.GetProp("_CIPCode")
                if atom.HasProp("_CIPCode")
                else None
            )

            if cip == "R":
                labels[idx] = "R"

            elif cip == "S":
                labels[idx] = "S"

            elif cip == "?":
                labels[idx] = "U"

            else:
                labels[idx] = "N"


        # Same q0, q1, q2 definition as the main calculation

        q0 = 0.0
        q1 = 0.0
        q2 = 0.0

        for i, lab in enumerate(labels):

            x = float(i + 1)

            s = SCORE_MAP.get(
                lab,
                0.0
            )

            q0 += s

            if lab in ("R", "S", "U"):
                q1 += x

            q2 += x * s


        if q0 == 0.0 or q1 == 0.0:
            return 0.0

        return 1.0 / (
            1.0 +
            np.exp(-(q2 / q1))
        )

    except Exception:
        return 0.25


# Read benchmark
df = pd.read_csv(input_file)

# Calculate f_chiral
values = []

for smiles in df["SMILES"]:

    values.append(
        calculate_f_chiral(smiles)
    )


# Save results
output = pd.DataFrame({
    "SMILES": df["SMILES"],
    "f_chiral": values
})

output.to_csv(
    output_file,
    index=False
)

print(
    "RDKit_VERSION="
    + rdBase.rdkitVersion
)

print(
    "N_MOLECULES="
    + str(len(output))
)
'''


with open(
    script_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(worker_script)


# ------------------------------------------------------------------------------
# 6. Calculate reference/current RDKit
# ------------------------------------------------------------------------------

print("\n")
print("=" * 70)
print(
    f"REFERENCE RDKit {current_rdkit_version}"
)
print("=" * 70)


# IMPORTANT:
# Define the function locally so this block is self-contained.

SCORE_MAP_MAIN = {
    "R": 1.0,
    "S": 0.5,
    "U": 0.75,
    "N": 0.0
}


def calculate_reference_f_chiral(smiles):

    try:

        mol = Chem.MolFromSmiles(
            smiles,
            sanitize=True
        )

        if mol is None:
            return 0.25

        Chem.AssignStereochemistry(
            mol,
            cleanIt=True,
            force=True,
            flagPossibleStereoCenters=True
        )

        labels = ["N"] * mol.GetNumAtoms()

        chiral_list = Chem.FindMolChiralCenters(
            mol,
            includeUnassigned=True,
            useLegacyImplementation=False
        )

        for idx, _ in chiral_list:

            atom = mol.GetAtomWithIdx(idx)

            cip = (
                atom.GetProp("_CIPCode")
                if atom.HasProp("_CIPCode")
                else None
            )

            if cip == "R":
                labels[idx] = "R"

            elif cip == "S":
                labels[idx] = "S"

            elif cip == "?":
                labels[idx] = "U"

            else:
                labels[idx] = "N"


        q0 = 0.0
        q1 = 0.0
        q2 = 0.0

        for i, lab in enumerate(labels):

            x = float(i + 1)

            s = SCORE_MAP_MAIN.get(
                lab,
                0.0
            )

            q0 += s

            if lab in ("R", "S", "U"):
                q1 += x

            q2 += x * s


        if q0 == 0.0 or q1 == 0.0:
            return 0.0

        return 1.0 / (
            1.0 +
            np.exp(-(q2 / q1))
        )

    except Exception:

        return 0.25


reference_values = []

for smiles in benchmark_smiles:

    reference_values.append(
        calculate_reference_f_chiral(smiles)
    )


comparison_df = pd.DataFrame({
    "SMILES": benchmark_smiles,
    f"f_chiral_{current_rdkit_version}":
        reference_values
})


# ------------------------------------------------------------------------------
# 7. Run each alternative RDKit version independently
# ------------------------------------------------------------------------------

for version in alternative_versions:

    print("\n")
    print("=" * 70)
    print(
        f"TESTING RDKit {version}"
    )
    print("=" * 70)


    install_dir = (
        "/tmp/rdkit_isolated_"
        + version.replace(".", "_")
    )

    output_file = (
        "/kaggle/working/"
        "f_chiral_rdkit_"
        + version.replace(".", "_")
        + ".csv"
    )


    # --------------------------------------------------------------------------
    # Install isolated RDKit version
    # --------------------------------------------------------------------------

    print(
        f"\nInstalling isolated RDKit: {version}"
    )

    if os.path.exists(install_dir):

        import shutil

        shutil.rmtree(
            install_dir
        )

    os.makedirs(
        install_dir,
        exist_ok=True
    )


    install_command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--target",
        install_dir,
        f"rdkit=={version}"
    ]


    result = subprocess.run(
        install_command,
        capture_output=True,
        text=True
    )


    if result.returncode != 0:

        print(
            f"ERROR installing RDKit {version}"
        )

        print(result.stderr)

        continue


    print(
        "Installation completed."
    )


    # --------------------------------------------------------------------------
    # Independent Python process
    # --------------------------------------------------------------------------

    env = os.environ.copy()

    env["PYTHONPATH"] = (
        install_dir
        + os.pathsep
        + env.get("PYTHONPATH", "")
    )


    command = [
        sys.executable,
        script_file,
        benchmark_file,
        output_file
    ]


    print(
        "Running independent Python process..."
    )


    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        env=env
    )


    print(result.stdout)


    if result.returncode != 0:

        print(
            f"ERROR running RDKit {version}"
        )

        print(result.stderr)

        continue


    # --------------------------------------------------------------------------
    # Read results
    # --------------------------------------------------------------------------

    alt_df = pd.read_csv(
        output_file
    )


    comparison_df[
        f"f_chiral_{version}"
    ] = alt_df["f_chiral"].values


# ------------------------------------------------------------------------------
# 8. Pairwise comparison
# ------------------------------------------------------------------------------

reference_column = (
    f"f_chiral_{current_rdkit_version}"
)

pairwise_results = []


for version in alternative_versions:

    column = (
        f"f_chiral_{version}"
    )

    if column not in comparison_df.columns:
        continue


    x = comparison_df[column].astype(float).values
    y = comparison_df[reference_column].astype(float).values

    abs_diff = np.abs(
        x - y
    )


    if (
        np.std(x) == 0
        or
        np.std(y) == 0
    ):

        pearson_r = np.nan

    else:

        pearson_r = np.corrcoef(
            x,
            y
        )[0, 1]


    pairwise_results.append({

        "RDKit_version":
            version,

        "reference_version":
            current_rdkit_version,

        "n_molecules":
            len(comparison_df),

        "Pearson_r":
            pearson_r,

        "MAE":
            np.mean(abs_diff),

        "Median_abs_difference":
            np.median(abs_diff),

        "Maximum_abs_difference":
            np.max(abs_diff)
    })


summary_df = pd.DataFrame(
    pairwise_results
)


# ------------------------------------------------------------------------------
# 9. Cross-version range
# ------------------------------------------------------------------------------

f_columns = [
    c
    for c in comparison_df.columns
    if c.startswith("f_chiral_")
]


comparison_df[
    "cross_version_range"
] = (
    comparison_df[f_columns].max(axis=1)
    -
    comparison_df[f_columns].min(axis=1)
)


# ------------------------------------------------------------------------------
# 10. Print results
# ------------------------------------------------------------------------------

print("\n")
print("=" * 75)
print(
    "RDKIT MULTI-VERSION ROBUSTNESS BENCHMARK"
)
print("=" * 75)

print(
    "\nCurrent/reference RDKit:",
    current_rdkit_version
)

print(
    "Molecules compared:",
    len(comparison_df)
)

print(
    "\nPairwise comparison against "
    f"RDKit {current_rdkit_version}:\n"
)

print(
    summary_df.to_string(
        index=False
    )
)


print("\n")
print("=" * 75)
print(
    "CROSS-VERSION RANGE"
)
print("=" * 75)

print(
    "Mean range:   "
    f"{comparison_df['cross_version_range'].mean():.6e}"
)

print(
    "Median range: "
    f"{comparison_df['cross_version_range'].median():.6e}"
)

print(
    "Maximum range:"
    f"{comparison_df['cross_version_range'].max():.6e}"
)


# ------------------------------------------------------------------------------
# 11. Save results
# ------------------------------------------------------------------------------

comparison_file = (
    "/kaggle/working/"
    "rdkit_4version_f_chiral_comparison.csv"
)

summary_file = (
    "/kaggle/working/"
    "rdkit_4version_f_chiral_summary.csv"
)

comparison_df.to_csv(
    comparison_file,
    index=False
)

summary_df.to_csv(
    summary_file,
    index=False
)


print("\nDetailed comparison saved to:")
print(comparison_file)

print("\nSummary statistics saved to:")
print(summary_file)


# ------------------------------------------------------------------------------
# 12. Interpretation
# ------------------------------------------------------------------------------

if (
    comparison_df[
        "cross_version_range"
    ].max() == 0
):

    print("\nInterpretation:")
    print(
        "f_chiral is identical across all "
        "tested RDKit versions."
    )

else:

    print("\nInterpretation:")
    print(
        "f_chiral shows measurable "
        "cross-version variation."
    )

Current Python version:
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Current RDKit version:
2026.03.6

RDKit versions selected for comparison:
 - 2024.3.1
 - 2024.9.1
 - 2025.3.1

Benchmark molecules exported: 500


REFERENCE RDKit 2026.03.6


TESTING RDKit 2024.3.1

Installing isolated RDKit: 2024.3.1
Installation completed.
Running independent Python process...
RDKit_VERSION=2024.03.1
N_MOLECULES=500



TESTING RDKit 2024.9.1

Installing isolated RDKit: 2024.9.1
Installation completed.
Running independent Python process...
RDKit_VERSION=2024.09.1
N_MOLECULES=500



TESTING RDKit 2025.3.1

Installing isolated RDKit: 2025.3.1
Installation completed.
Running independent Python process...
RDKit_VERSION=2025.03.1
N_MOLECULES=500



RDKIT MULTI-VERSION ROBUSTNESS BENCHMARK

Current/reference RDKit: 2026.03.6
Molecules compared: 500

Pairwise comparison against RDKit 2026.03.6:

RDKit_version reference_version  n_molecules  Pearson_r  MAE  Median_abs_difference  Maximum_abs_differenc

In [8]:
# ==============================================================================
# BLOCK 7 — FINAL VALIDATION OF CROSS-VERSION ROBUSTNESS
# ==============================================================================

import os
import pandas as pd
import numpy as np

comparison_file = (
    "/kaggle/working/"
    "rdkit_4version_f_chiral_comparison.csv"
)

summary_file = (
    "/kaggle/working/"
    "rdkit_4version_f_chiral_summary.csv"
)

print("=" * 75)
print("FINAL VALIDATION OF RDKit CROSS-VERSION ROBUSTNESS")
print("=" * 75)

# ------------------------------------------------------------------------------
# 1. Load detailed comparison
# ------------------------------------------------------------------------------

if not os.path.exists(comparison_file):
    raise FileNotFoundError(
        f"Comparison file not found:\n{comparison_file}\n"
        "Run BLOCK 6 first."
    )

comparison_df = pd.read_csv(comparison_file)

print("\nLoaded comparison file:")
print(comparison_file)

print("\nRows:", len(comparison_df))
print("Columns:")
for col in comparison_df.columns:
    print(" -", col)


# ------------------------------------------------------------------------------
# 2. Identify RDKit f_chiral columns
# ------------------------------------------------------------------------------

f_columns = [
    col for col in comparison_df.columns
    if "f_chiral" in col.lower()
]

print("\nf_chiral-related columns detected:")
for col in f_columns:
    print(" -", col)


# ------------------------------------------------------------------------------
# 3. Exact numerical comparison
# ------------------------------------------------------------------------------

if len(f_columns) < 2:
    raise ValueError(
        "At least two f_chiral columns are required "
        "for cross-version validation."
    )

f_values = comparison_df[f_columns].astype(float)

max_pairwise_difference = 0.0
comparison_pairs = []

for i in range(len(f_columns)):
    for j in range(i + 1, len(f_columns)):

        col_a = f_columns[i]
        col_b = f_columns[j]

        abs_diff = np.abs(
            f_values[col_a] - f_values[col_b]
        )

        pair_max = abs_diff.max()
        pair_mean = abs_diff.mean()
        pair_median = abs_diff.median()

        max_pairwise_difference = max(
            max_pairwise_difference,
            pair_max
        )

        comparison_pairs.append({
            "comparison": f"{col_a} vs {col_b}",
            "mean_abs_difference": pair_mean,
            "median_abs_difference": pair_median,
            "maximum_abs_difference": pair_max,
            "n_exactly_equal": int((abs_diff == 0).sum()),
            "n_molecules": len(abs_diff)
        })


pairwise_df = pd.DataFrame(comparison_pairs)

print("\n" + "=" * 75)
print("PAIRWISE EXACT COMPARISON")
print("=" * 75)

print(pairwise_df.to_string(index=False))


# ------------------------------------------------------------------------------
# 4. Molecule-level cross-version range
# ------------------------------------------------------------------------------

comparison_df["validation_cross_version_range"] = (
    f_values.max(axis=1) -
    f_values.min(axis=1)
)

range_series = comparison_df[
    "validation_cross_version_range"
]

print("\n" + "=" * 75)
print("MOLECULE-LEVEL CROSS-VERSION RANGE")
print("=" * 75)

print(
    "Mean range:   "
    f"{range_series.mean():.12e}"
)

print(
    "Median range: "
    f"{range_series.median():.12e}"
)

print(
    "Maximum range:"
    f"{range_series.max():.12e}"
)

print(
    "Molecules with non-zero range:",
    int((range_series != 0).sum())
)

print(
    "Molecules with exactly zero range:",
    int((range_series == 0).sum())
)


# ------------------------------------------------------------------------------
# 5. Exact equality test
# ------------------------------------------------------------------------------

all_exactly_equal = (
    max_pairwise_difference == 0.0
)

print("\n" + "=" * 75)
print("EXACT EQUALITY TEST")
print("=" * 75)

if all_exactly_equal:

    print(
        "PASS: All f_chiral values are exactly identical "
        "across all tested RDKit versions."
    )

else:

    print(
        "RESULT: Differences were detected between "
        "at least some RDKit versions."
    )


# ------------------------------------------------------------------------------
# 6. Save final validation tables
# ------------------------------------------------------------------------------

pairwise_validation_file = (
    "/kaggle/working/"
    "rdkit_cross_version_pairwise_validation.csv"
)

molecule_validation_file = (
    "/kaggle/working/"
    "rdkit_cross_version_molecule_validation.csv"
)

pairwise_df.to_csv(
    pairwise_validation_file,
    index=False
)

comparison_df.to_csv(
    molecule_validation_file,
    index=False
)


# ------------------------------------------------------------------------------
# 7. Final interpretation
# ------------------------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL INTERPRETATION")
print("=" * 75)

if all_exactly_equal:

    print(
        "For the 500-molecule benchmark, f_chiral was "
        "numerically identical across RDKit 2024.3.1, "
        "2024.9.1, 2025.3.1, and 2026.03.6."
    )

    print(
        "\nThis supports empirical robustness of the "
        "implemented f_chiral calculation to RDKit "
        "version changes within the tested versions."
    )

else:

    print(
        "The benchmark detected measurable differences "
        "between at least some tested RDKit versions."
    )

print("\nValidation files saved:")
print(pairwise_validation_file)
print(molecule_validation_file)

FINAL VALIDATION OF RDKit CROSS-VERSION ROBUSTNESS

Loaded comparison file:
/kaggle/working/rdkit_4version_f_chiral_comparison.csv

Rows: 500
Columns:
 - SMILES
 - f_chiral_2026.03.6
 - f_chiral_2024.3.1
 - f_chiral_2024.9.1
 - f_chiral_2025.3.1
 - cross_version_range

f_chiral-related columns detected:
 - f_chiral_2026.03.6
 - f_chiral_2024.3.1
 - f_chiral_2024.9.1
 - f_chiral_2025.3.1

PAIRWISE EXACT COMPARISON
                             comparison  mean_abs_difference  median_abs_difference  maximum_abs_difference  n_exactly_equal  n_molecules
f_chiral_2026.03.6 vs f_chiral_2024.3.1                  0.0                    0.0                     0.0              500          500
f_chiral_2026.03.6 vs f_chiral_2024.9.1                  0.0                    0.0                     0.0              500          500
f_chiral_2026.03.6 vs f_chiral_2025.3.1                  0.0                    0.0                     0.0              500          500
 f_chiral_2024.3.1 vs f_chiral_